# SWEEP-Net Milestone 2: Decoding the Domain Gap and Temporal Gaslighting

## 1. Introduction & Objective Recap
In Milestone 1, we established the foundation of **SWEEP-Net**, a spatiotemporal encoder (Atrous-MobileNet with ODConv and TSM) designed to translate Continuous Wavelet Transform (CWT) EEG topomaps into a 128D emotional latent space. Our ultimate goal remains the creation of an energy-efficient Spiking Neural Network (SNN) for Brain-Computer Interfaces (BCI).

However, in Milestone 2, we confront the fundamental bottleneck of EEG Emotion Recognition: **The Cross-Subject Domain Gap and Label Noise.** Standard contrastive learning models suffer from severe "shortcut learning," where the network memorizes a subject's biometric skull geometry and baseline rhythms rather than the underlying emotion. 

This notebook presents a complete ablation study of three major experimental iterations to destroy this biometric shortcut, diagnose the physiological reality of the dataset, and isolate the true semantic geometry of human emotion.

## 2. Comprehensive Implementation Inventory

To attack the domain gap and temporal gaslighting, we successfully implemented the following custom components into our codebase:

**Architecture & Optimizers:**
*   **ODConv (Omni-Dimensional Dynamic Convolutions):** Replaced static depthwise filters with 4-expert dynamic routing to adapt to transient emotional topologies.
*   **Muon Hybrid Optimizer:** Implemented the Newton-Schulz second-order orthogonalized optimizer for pointwise convolutions/heads, paired with AdamW for depthwise layers.

**Loss Functions & Contrastive Mechanics:**
*   **SupMoCo (Supervised Momentum Contrast):** Utilizing a fast-updating queue (`m=0.9, size=512`).
*   **DCL (Decoupled Contrastive Learning):** Removing positive pairs from the denominator to maintain continuous gradient pull and prevent early preemption.
*   **1 + Dice Score Penalty:** Topology-weighted hard negative mining to violently repel overlapping emotional ground truths.
*   **SCDA (Supervised Contrastive Domain Adaptation):** Modifying the InfoNCE loss to actively penalize intra-subject clustering.
*   **CosFace Angular Margin:** Injecting an angular margin into the contrastive logits to enforce strict intra-emotion diversity radiuses.
*   **Temporal Queue Decay:** Dynamically decaying the weight of stale queue embeddings over time.
*   **cMMD (Class-Conditional Maximum Mean Discrepancy):** Statistical alignment of subject distributions.
*   **DANN (Domain-Adversarial Neural Networks):** Gradient Reversal Layer (GRL) subject discriminator.

**Data-Level Augmentations & Preprocessing:**
*   **3D Cartesian Fourier Mixup:** Dynamically swapping the low-frequency amplitudes (skull) of batch subjects directly in the complex Cartesian domain to avoid phase-wrapping artifacts.
*   **Session-Level Euclidean Alignment (EA):** Raw sequence covariance whitening to map all spatial topologies to the Identity matrix.

According to the feedback provided to previous milestone, we leveraged Hydra and OmegaConf due to their modularized configuration approach, which eases experiments and ablation.

To delinate the metrics used to quantify success and shortcomings, we dedicated a list. We also want to emphasize that these depend on feature vectors received from backbone directly, since `ProjectionHead` squashes all parts responsible for classification to satisfy contrastive loss:

*   **Proxy-A Subject ID Distance:** Trains `Linear SVC` on the features to classify Subject IDs. While, usually `RBF kernel` is preferred due to nonlinearity, it is computationally heavy. Since the encoder itself is deep enough to provide necessary nonlinearity, the choice should not reflect any substantial weakness.

*   **4-Way Linear Probes:** The most prominent and established way to measure success of contrastive learning is to fit a "dumb" linear head (via L-BGFS, to eschew the iterative approach via backpropagation). We extend it to capture 4 different directions: `train -> train`, `val -> val`, `train -> val`, `val -> train`. The former two are fitted via Group 5-Fold based on trial (each time any subjects watches any video), while the latter two are trained as a whole: `train -> val` is cross-subject generalization---i.e., train on TRAIN set (Subjects 2-16), test on VAL set (Leave-One-Subject-Out 1)---and `val -> train` is reverse-subject generalization.

* **UMAPs:** We calculate `Validation separability` UMAP (fit val, transform val), `Domain Shift` (fit train, transform both train (as smooth points) and val (stars)), and `Subject Font` (fit train, transform train to show subject IDs).

* **Intra-Bag Cosine Similarity Map:** We sample a random bag (=trial) that has enough windows (>100) and is not neutral emotion (since it is equivalent to baseline mood, in contrary to assumption that neutral-labeled trials are also presenting cognitive load). Next, we calculate cosine similarity between features of any two windows.

## 3. Limitations & Failures: The "Alien Skull" and the 99% Cheat Code

Despite the extensive implementation list, tracking offline **Proxy-A Subject ID Probes** revealed that the network continuously found ways to memorize the subjects (hitting >90% Subject ID accuracy). We documented several theoretical failures that forced us to bench certain SOTA techniques:

1. **DANN + Muon Hybrid = Mathematical Explosion:** We attempted to combine Gradient Reversal (DANN) with the Muon optimizer. **Failure:** DANN creates a min-max adversarial saddle point. Muon is a second-order optimizer looking for convex bowls. The Newton-Schulz iterations mathematically inverted the Hessian, accelerating the gradients to $7 \times 10^{15}$ and causing catastrophic `NaN` collapse. Muon and DANN were benched for Phase 1A.
2. **cMMD = Zero-Variance Trap:** We attempted to statistically pull subject centroids together. **Failure:** The network realized the easiest way to minimize the MMD distance between subjects was to output the exact same constant 128D vector for every image, collapsing the entire hypersphere.
3. **3D Cartesian Fourier Mixup = Post-CWT Bottleneck:** We attempted to dynamically swap the physical skull using FFTs. **Failure:** Because CWT is a non-linear transformation, the spatial covariance (impedance) was already multiplicatively baked into the high and low frequencies. Mixing post-CWT was too late to hide the biometric fingerprint.
4. **CosFace Margin & Temporal Queue Decay = Over-engineering:** Temporal decay was rendered dormant due to our highly restricted queue size (512). The CosFace margin actively punished the network during the early, fragile epochs, exacerbating representation collapse. Both were disabled.

These failures yielded a critical conclusion: **We cannot solve the spatial domain gap using loss functions or post-CWT augmentations.** We must solve it at the raw data level.


## Visualization tools

In order to present the results, we have to fetch them from W&B logging server.

In [ ]:
import os, tempfile
from PIL import Image as PILImage
from IPython.display import display
import wandb

In [ ]:

WANDB_ENTITY = "emammadli"
WANDB_PROJECT = "snn-encoding-showdown"

# SupMoCo, Partial BN, DCL x (1+Dice) negative weighting
ex1 = "7gp9y8bk"
ex1_eval = "9aupf4hd

# + SCDA, + LOTO masking
ex2 = "0vhic8qr"
ex2_eval = "05ec57zj"

# + EA, + LOTO masking
ex3 = ["fubt6gwv", "7bfkrnq5"]
ex3_eval = "tvsp446p"


api = wandb.Api()

In [ ]:

def fetch_loss_history(entity, project, run_id):
    run = api.run(f"{entity}/{project}/{run_id}")
    
    history = run.history(keys=["Phase1/Train/Loss"], pandas=True)
    return history


def fetch_metric(entity, project, run_id, key):
    """Fetch a single scalar metric's full history as a pandas Series."""
    run = api.run(f"{entity}/{project}/{run_id}")
    history = run.history(keys=[key], pandas=True)
    return history[key].dropna()

def fetch_accuracy(entity, project, run_id, key="Phase1a/Val/Emotion_Probe_Val_Acc"):
    run = api.run(f"{entity}/{project}/{run_id}")
    # First try the run summary (fastest)
    if key in run.summary:
        return run.summary[key]
    # Fallback: pull full history and take the last value
    history = run.history(keys=[key], pandas=True)
    series = history[key].dropna()
    if len(series) == 0:
        raise KeyError(f"Metric \"{key}\" not found in run {run_id}")
    return series.iloc[-1]

def fetch_best_metric(entity, project, run_id, key, mode="max"):
    series = fetch_metric(entity, project, run_id, key)
    return series.max() if mode == "max" else series.min()

def fetch_images(entity, project, run_id, key_prefix="Visuals", step=None):
    run = api.run(f"{entity}/{project}/{run_id}")
    images = []
    
    for f in run.files():
        if f.name.startswith("media/images") and key_prefix.lower() in f.name.lower():
            tmp = tempfile.mktemp(suffix=".png")
            f.download(root=tempfile.gettempdir(), replace=True)
            dl_path = os.path.join(tempfile.gettempdir(), f.name)
            if os.path.exists(dl_path):
                img = PILImage.open(dl_path)
                images.append((f.name, img))
    
    if not images:
        history = run.history(pandas=True)
        media_cols = [c for c in history.columns if key_prefix in c]
        print(f"No image files found. Available media columns: {media_cols}")
    
    return images

def show_run_images(entity, project, run_id, key_prefix="Visuals", max_images=8, figsize=(16, 8)):
    """Fetch images from a W&B run and display them in a matplotlib grid."""
    imgs = fetch_images(entity, project, run_id, key_prefix)
    if not imgs:
        print("No images found.")
        return
    
    imgs = imgs[:max_images]
    n = len(imgs)
    cols = min(4, n)
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=figsize)
    if n == 1:
        axes = [axes]
    else:
        axes = axes.flat
    for ax, (name, img) in zip(axes, imgs):
        ax.imshow(img)
        ax.set_title(os.path.basename(name), fontsize=8)
        ax.axis("off")
    for ax in list(axes)[n:]:
        ax.axis("off")
    plt.tight_layout()
    plt.show()

def fetch_probe_history(entity, project, run_id):
    """Return a DataFrame with all offline-probe metrics over time."""
    keys = [
        "Phase1a/Train/Emotion_Probe_CV_Acc",
        "Phase1a/Val/Emotion_Probe_Val_Acc",
        "Phase1a/Train/Subject_Probe_CV_Acc",
        "Phase1a/Val/Subject_Probe_Val_Acc",
    ]
    run = api.run(f"{entity}/{project}/{run_id}")
    history = run.history(keys=keys, pandas=True)
    return history.dropna(how="all")

def print_run_summary(entity, project, run_id, keys=None):
    run = api.run(f"{entity}/{project}/{run_id}")
    summary = dict(run.summary)
    if keys:
        summary = {k: v for k, v in summary.items() if k in keys}
    
    print(f"╔══ Run: {run.name} ({run_id}) ══╗")
    for k, v in sorted(summary.items()):
        if isinstance(v, (int, float)):
            print(f"  {k:.<50s} {v:.4f}")
    print(f"╚{'═' * 42}╝")



In [ ]:
# == Multi-Run Fetching & Plotting Helpers ================================
# When a run was interrupted and resumed under a new run ID, pass all
# continuation IDs as a list.  These helpers concatenate the histories
# in order and produce seamless plots.
# =========================================================================

import pandas as pd

# ── Core: concatenate a metric across multiple resumed runs ──
def fetch_metric_multi(entity, project, run_ids, key):
    """Fetch a scalar metric from one or more runs and concatenate them.
    
    Parameters
    ----------
    run_ids : str | list[str]
        A single run ID or a list of run IDs in chronological order.
        When a list is given the histories are concatenated with a
        continuous step index.
    key : str
        The W&B metric key, e.g. "Phase1a/Train/Loss".
    
    Returns
    -------
    pd.Series with a continuous integer index (step).
    """
    if isinstance(run_ids, str):
        run_ids = [run_ids]
    
    parts = []
    offset = 0
    for rid in run_ids:
        run = api.run(f"{entity}/{project}/{rid}")
        h = run.history(keys=[key], pandas=True)
        series = h[key].dropna().reset_index(drop=True)
        series.index = series.index + offset
        parts.append(series)
        offset += len(series)
    
    return pd.concat(parts)


def fetch_accuracy_multi(entity, project, run_ids, key="Phase1a/Val/Emotion_Probe_Val_Acc"):
    """Return the final value of a metric across chained runs (last run's last value)."""
    if isinstance(run_ids, str):
        run_ids = [run_ids]
    return fetch_accuracy(entity, project, run_ids[-1], key=key)


# ── Plotting: single metric curve ──
def plot_metric(entity, project, run_ids, key, ax=None,
               label=None, color=None, title=None, ylabel=None,
               smooth_window=0, grid=True):
    """Plot a single metric from one or more chained runs.
    
    Parameters
    ----------
    run_ids : str | list[str]
        Single run ID or list of chained run IDs.
    smooth_window : int
        If > 0, apply a rolling average for visual clarity.
    """
    series = fetch_metric_multi(entity, project, run_ids, key)
    
    if ax is None:
        fig, ax = plt.subplots(figsize=(10, 4))
    
    plot_label = label or key.split("/")[-1]
    
    if smooth_window > 0:
        smoothed = series.rolling(smooth_window, min_periods=1).mean()
        ax.plot(smoothed.index, smoothed.values, label=plot_label, color=color, linewidth=2)
        ax.plot(series.index, series.values, alpha=0.15, color=color, linewidth=0.8)
    else:
        ax.plot(series.index, series.values, label=plot_label, color=color, linewidth=1.5)
    
    ax.set_xlabel("Step")
    ax.set_ylabel(ylabel or key.split("/")[-1])
    if title:
        ax.set_title(title)
    if grid:
        ax.grid(True, alpha=0.3)
    ax.legend()
    return ax


# ── Plotting: loss curves (train + val side by side or overlaid) ──
def plot_loss(entity, project, run_ids, phase="1a",
             smooth_window=5, figsize=(14, 5)):
    """Plot Train & Val loss for a (potentially multi-run) experiment.
    
    Parameters
    ----------
    run_ids : str | list[str]
        Single or chained run IDs.
    phase : str
        Phase prefix (e.g. "1a", "2", "3").
    """
    train_key = f"Phase{phase}/Train/Loss"
    val_key   = f"Phase{phase}/Val/Loss"
    
    train = fetch_metric_multi(entity, project, run_ids, train_key)
    
    # Val loss may not exist for Phase 1A contrastive runs
    try:
        val = fetch_metric_multi(entity, project, run_ids, val_key)
        has_val = len(val) > 0
    except Exception:
        has_val = False
    
    fig, ax = plt.subplots(figsize=figsize)
    
    if smooth_window > 0:
        sm = train.rolling(smooth_window, min_periods=1).mean()
        ax.plot(sm.index, sm.values, label="Train Loss", color="#2196F3", linewidth=2)
        ax.plot(train.index, train.values, alpha=0.12, color="#2196F3", linewidth=0.8)
    else:
        ax.plot(train.index, train.values, label="Train Loss", color="#2196F3", linewidth=1.5)
    
    if has_val:
        if smooth_window > 0:
            sm_v = val.rolling(smooth_window, min_periods=1).mean()
            ax.plot(sm_v.index, sm_v.values, label="Val Loss", color="#FF5722", linewidth=2)
            ax.plot(val.index, val.values, alpha=0.12, color="#FF5722", linewidth=0.8)
        else:
            ax.plot(val.index, val.values, label="Val Loss", color="#FF5722", linewidth=1.5)
    
    ax.set_xlabel("Step")
    ax.set_ylabel("Loss")
    ax.set_title(f"Phase {phase} Loss Curve")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


# ── Plotting: probe accuracy curves (emotion + subject) ──
def plot_probes(entity, project, run_ids, figsize=(14, 5)):
    """Plot Emotion & Subject probe accuracies over training.
    
    Shows Emotion Probe CV Acc vs Subject Probe CV Acc (Proxy-A)
    to visualize the privacy-utility tradeoff.
    """
    emo_key  = "Phase1a/Train/Emotion_Probe_CV_Acc"
    subj_key = "Phase1a/Train/Subject_Probe_CV_Acc"
    
    emo  = fetch_metric_multi(entity, project, run_ids, emo_key)
    subj = fetch_metric_multi(entity, project, run_ids, subj_key)
    
    fig, ax = plt.subplots(figsize=figsize)
    
    ax.plot(emo.index, emo.values, label="Emotion Probe CV Acc", 
            color="#4CAF50", linewidth=2, marker="o", markersize=4)
    ax.plot(subj.index, subj.values, label="Subject Probe CV Acc (Proxy-A)", 
            color="#F44336", linewidth=2, marker="s", markersize=4)
    
    # Chance line for subject ID (if we know # subjects)
    ax.axhline(y=0.5, color="gray", linestyle="--", alpha=0.4, label="Chance (2-class)")
    
    ax.set_xlabel("Probe Step")
    ax.set_ylabel("Balanced Accuracy")
    ax.set_title("Privacy-Utility Tradeoff: Emotion vs Subject Probes")
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.set_ylim(0, 1.05)
    plt.tight_layout()
    plt.show()


# ── Plotting: val probes (cross-subject generalization) ──
def plot_val_probes(entity, project, run_ids, figsize=(14, 5)):
    """Plot Val Emotion Probe vs Val Subject Probe (cross-subject eval)."""
    emo_key  = "Phase1a/Val/Emotion_Probe_Val_Acc"
    subj_key = "Phase1a/Val/Subject_Probe_Val_Acc"
    
    emo  = fetch_metric_multi(entity, project, run_ids, emo_key)
    subj = fetch_metric_multi(entity, project, run_ids, subj_key)
    
    fig, ax = plt.subplots(figsize=figsize)
    
    ax.plot(emo.index, emo.values, label="Emotion Val Acc (Zero-Shot)", 
            color="#2196F3", linewidth=2, marker="o", markersize=4)
    ax.plot(subj.index, subj.values, label="Subject Val Acc (Proxy-A Cross)", 
            color="#FF9800", linewidth=2, marker="s", markersize=4)
    
    ax.set_xlabel("Probe Step")
    ax.set_ylabel("Balanced Accuracy")
    ax.set_title("Validation Probes: Emotion vs Subject (Cross-Subject)")
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.set_ylim(0, 1.05)
    plt.tight_layout()
    plt.show()


# ── Plotting: compare multiple experiments on the same axis ──
def plot_compare(entity, project, experiments, key,
                 smooth_window=5, figsize=(12, 5), title=None):
    """Overlay the same metric from multiple experiments for comparison.
    
    Parameters
    ----------
    experiments : dict[str, str | list[str]]
        Mapping of experiment_name -> run_id(s).
        Example: {"Exp1 (SCDA)": "abc123", "Exp3 (EA)": ["def456", "ghi789"]}
    key : str
        The metric key to compare.
    """
    fig, ax = plt.subplots(figsize=figsize)
    colors = ["#2196F3", "#4CAF50", "#FF5722", "#9C27B0", "#FF9800", "#00BCD4"]
    
    for i, (name, rids) in enumerate(experiments.items()):
        color = colors[i % len(colors)]
        series = fetch_metric_multi(entity, project, rids, key)
        if smooth_window > 0:
            sm = series.rolling(smooth_window, min_periods=1).mean()
            ax.plot(sm.index, sm.values, label=name, color=color, linewidth=2)
            ax.plot(series.index, series.values, alpha=0.1, color=color, linewidth=0.8)
        else:
            ax.plot(series.index, series.values, label=name, color=color, linewidth=1.5)
    
    ax.set_xlabel("Step")
    ax.set_ylabel(key.split("/")[-1])
    ax.set_title(title or f"Comparison: {key}")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


# ── Plotting: full dashboard for a single experiment ──
def plot_dashboard(entity, project, run_ids, phase="1a",
                   smooth_window=5, figsize=(16, 10)):
    """All-in-one dashboard: Loss + Emotion Probe + Subject Probe + LR.
    
    Parameters
    ----------
    run_ids : str | list[str]
        Single or chained run IDs.
    """
    fig, axes = plt.subplots(2, 2, figsize=figsize)
    
    # Top-left: Train Loss
    plot_metric(entity, project, run_ids,
               f"Phase{phase}/Train/Loss",
               ax=axes[0, 0], label="Train Loss", color="#2196F3",
               title="Train Loss", ylabel="Loss",
               smooth_window=smooth_window)
    
    # Top-right: Emotion Probe
    try:
        plot_metric(entity, project, run_ids,
                   "Phase1a/Train/Emotion_Probe_CV_Acc",
                   ax=axes[0, 1], label="Emotion CV", color="#4CAF50",
                   title="Emotion Probe", ylabel="Accuracy")
        plot_metric(entity, project, run_ids,
                   "Phase1a/Val/Emotion_Probe_Val_Acc",
                   ax=axes[0, 1], label="Emotion Val", color="#81C784")
    except Exception:
        axes[0, 1].text(0.5, 0.5, "No probe data", ha="center", va="center",
                        transform=axes[0, 1].transAxes)
    
    # Bottom-left: Subject Probe (Proxy-A)
    try:
        plot_metric(entity, project, run_ids,
                   "Phase1a/Train/Subject_Probe_CV_Acc",
                   ax=axes[1, 0], label="Subject CV (Proxy-A)", color="#F44336",
                   title="Subject Probe (Proxy-A)", ylabel="Accuracy")
        plot_metric(entity, project, run_ids,
                   "Phase1a/Val/Subject_Probe_Val_Acc",
                   ax=axes[1, 0], label="Subject Val", color="#EF9A9A")
    except Exception:
        axes[1, 0].text(0.5, 0.5, "No probe data", ha="center", va="center",
                        transform=axes[1, 0].transAxes)
    
    # Bottom-right: LR schedule
    try:
        plot_metric(entity, project, run_ids,
                   "LR", ax=axes[1, 1], label="Learning Rate", color="#9C27B0",
                   title="Learning Rate Schedule", ylabel="LR")
    except Exception:
        axes[1, 1].text(0.5, 0.5, "No LR data", ha="center", va="center",
                        transform=axes[1, 1].transAxes)
    
    plt.suptitle(f"Phase {phase} Dashboard", fontsize=14, fontweight="bold", y=1.01)
    plt.tight_layout()
    plt.show()



## 4. Experiment 1: Pure SupMoCo + DCL (The Baseline)
**Goal:** Determine if a highly regularized contrastive encoder can naturally learn universal emotional representations without explicit domain-adaptation constraints.

**Method:** We implemented Supervised Momentum Contrast (SupMoCo) paired with Decoupled Contrastive Learning (DCL) to prevent gradient preemption. Hard negatives `(Diff Emotion, Same Subject)` were heavily penalized using a `1 + Dice Score` multiplier  to separate topologically overlapping emotions. No data-level (EA/FDA) or loss-level (SCDA) domain anonymization was applied.

Note: Here we neglected the usage of `BatchNorm` within `Attention` modules of `ODConv` and `ProjectionHead`, while leaving `LayerNorm` (implemented via `GroupNorm`) everywhere else for normalization. While intra-batch communication documented by He et al. (2020) is concern for momentum-contrast learning, where the framework cheats by using statistics shared across batch to figure out positive sample, it is most prominent within unsupervised learning since positive pair is augmented version of single sample. By *contrast*, in supervised learning, positive pair's definition is streched to entail all samples within same label, which reduces the risk of cheating in our case (+ `LayerNorm` is more used than `BatchNorm`). Yet, we resolved this issue in future experiments, and to maintain backward compatibility, we have `use_batchnorm` flag in config files.

In [ ]:
!python main.py --config config/archive_ablations/scda_partialbn_adamw.yaml --loso 1

In [ ]:
# ── Experiment 1: Loss Curve ──
plot_loss(WANDB_ENTITY, WANDB_PROJECT, ex1, phase='1a', smooth_window=5)

# ── Experiment 1: Linear Probe Accuracy (single data points) ──
ex1_lp_vv  = fetch_accuracy(WANDB_ENTITY, WANDB_PROJECT, ex1_eval, key='Eval/LP_ValVal_CV')
ex1_lp_vt  = fetch_accuracy(WANDB_ENTITY, WANDB_PROJECT, ex1_eval, key='Eval/LP_ValTrain')
ex1_lp_tv  = fetch_accuracy(WANDB_ENTITY, WANDB_PROJECT, ex1_eval, key='Eval/LP_TrainVal')
ex1_lp_tt  = fetch_accuracy(WANDB_ENTITY, WANDB_PROJECT, ex1_eval, key='Eval/LP_TrainTrain_CV')

print('\n' + '='*60)
print('  Experiment 1: Linear Probe Results')
print('='*60)
print(f'  Val->Val   (5-fold GroupCV):  {ex1_lp_vv:.4f}')
print(f'  Val->Train (cross-split):    {ex1_lp_vt:.4f}')
print(f'  Train->Val (cross-split):    {ex1_lp_tv:.4f}')
print(f'  Train->Train (5-fold CV):    {ex1_lp_tt:.4f}')


### Experiment 1 Results & Interpretation
*   **Zero-Shot (Train -> Val):** ~31%
*   **5 Group  Holdout (Val -> Val):** ~90%
*   **Proxy-A Subject ID Probe:** ~91% (The network successfully memorized the subjects).

**Interpretation:** The 90% `Val->Val` linear probe proves that the network *successfully encoded highly discriminative emotional geometry*, implying that Fead/Disgust dillema is partially solved. However, 


## 5. Experiment 2: The "Gaslighting" Discovery (Adding LOTO Masking)
**Goal:** In EEG datasets, a 60-second video labeled "Fear" contains maybe 5 seconds of actual climax, and 55 seconds of neutral staring. We hypothesize that 1-second labels are fundamentally "gaslighting" the contrastive loss.

**Method:** We introduced **Leave-One-Trial-Out (LOTO) Masking** to the denominator. We explicitly masked out intra-trial negatives to prevent the network from violently pushing a "Fear Climax" away from a "Fear Buildup" occurring in the same video. 



In [ ]:
# ── Experiment 2: Loss Curve ──
plot_loss(WANDB_ENTITY, WANDB_PROJECT, ex2, phase='1a', smooth_window=5)

# ── Experiment 2: Subject Probe ──
plot_metric(WANDB_ENTITY, WANDB_PROJECT, ex2,
            'Phase1a/Train/Subject_Probe_CV_Acc',
            label='Subject Probe CV Acc (Proxy-A)',
            color='#F44336', title='Exp 2: Subject Probe (Proxy-A)',
            ylabel='Accuracy')
plt.show()

# ── Experiment 2: Linear Probe Accuracy (single data points) ──
ex2_lp_vv  = fetch_accuracy(WANDB_ENTITY, WANDB_PROJECT, ex2_eval, key='Eval/LP_ValVal_CV')
ex2_lp_vt  = fetch_accuracy(WANDB_ENTITY, WANDB_PROJECT, ex2_eval, key='Eval/LP_ValTrain')
ex2_lp_tv  = fetch_accuracy(WANDB_ENTITY, WANDB_PROJECT, ex2_eval, key='Eval/LP_TrainVal')
ex2_lp_tt  = fetch_accuracy(WANDB_ENTITY, WANDB_PROJECT, ex2_eval, key='Eval/LP_TrainTrain_CV')

print('\n' + '='*60)
print('  Experiment 2: Linear Probe Results')
print('='*60)
print(f'  Val->Val   (5-fold GroupCV):  {ex2_lp_vv:.4f}')
print(f'  Val->Train (cross-split):    {ex2_lp_vt:.4f}')
print(f'  Train->Val (cross-split):    {ex2_lp_tv:.4f}')
print(f'  Train->Train (5-fold CV):    {ex2_lp_tt:.4f}')


To empirically prove this gaslighting, we generated an **Intra-Bag Cosine Similarity Heatmap** using the unregularized $h$ embeddings.


In [ ]:
# ── Experiment 2: Visualization Artifacts ──
show_run_images(WANDB_ENTITY, WANDB_PROJECT, ex2_eval,
                key_prefix='IntraBag_CosSim', max_images=1, figsize=(8, 7))

show_run_images(WANDB_ENTITY, WANDB_PROJECT, ex2_eval,
                key_prefix='UMAP3_ValSeparability', max_images=1, figsize=(10, 8))

show_run_images(WANDB_ENTITY, WANDB_PROJECT, ex2_eval,
                key_prefix='UMAP1_SubjectFont', max_images=1, figsize=(10, 8))

show_run_images(WANDB_ENTITY, WANDB_PROJECT, ex2_eval,
                key_prefix='UMAP2_DomainShift', max_images=1, figsize=(10, 8))


### Experiment 2 Results & Interpretation
*   **Result:** The network stalled for ~40 epochs before the loss snapped, indicating it struggled to find global semantic boundaries without the "easy" intra-trial negatives. 
*   **Visual Proof:** The heatmap visually proves the gaslighting phenomenon. We see massive, distinct blocks of time (e.g., seconds 0 to 180 forming a highly similar "buildup" block, and seconds 180-300 forming a distinct "climax" block). The 1-second labels are objectively noisy.


## 6. Experiment 3: The Breakthrough (Euclidean Alignment + DCL)
**Goal:** To permanently destroy the spatial biometric fingerprint before it enters the network, allowing pure contrastive learning to function properly.

**Method:** We applied **Session-Level Euclidean Alignment (EA)** offline to the raw 1D sequences before CWT. EA computes the mean spatial covariance matrix of a session and multiplies the raw data by its inverse square root ($R^{-1/2}$), forcing every session's baseline geometry to the exact same Identity Matrix. Because EA physically erased the subjects, we dropped SCDA (as it creates a mathematical paradox on anonymized data) and reverted to pure DCL + LOTO.


In [ ]:
# ── Experiment 3: Loss Curve (multi-run: chained) ──
plot_loss(WANDB_ENTITY, WANDB_PROJECT, ex3, phase='1a', smooth_window=5)

# ── Experiment 3: Subject Probe ──
plot_metric(WANDB_ENTITY, WANDB_PROJECT, ex3,
            'Phase1a/Train/Subject_Probe_CV_Acc',
            label='Subject Probe CV Acc (Proxy-A)',
            color='#F44336', title='Exp 3: Subject Probe (Proxy-A)',
            ylabel='Accuracy')
plt.show()

# ── Experiment 3: Linear Probe Accuracy (single data points) ──
ex3_lp_vv  = fetch_accuracy(WANDB_ENTITY, WANDB_PROJECT, ex3_eval, key='Eval/LP_ValVal_CV')
ex3_lp_vt  = fetch_accuracy(WANDB_ENTITY, WANDB_PROJECT, ex3_eval, key='Eval/LP_ValTrain')
ex3_lp_tv  = fetch_accuracy(WANDB_ENTITY, WANDB_PROJECT, ex3_eval, key='Eval/LP_TrainVal')
ex3_lp_tt  = fetch_accuracy(WANDB_ENTITY, WANDB_PROJECT, ex3_eval, key='Eval/LP_TrainTrain_CV')

print('\n' + '='*60)
print('  Experiment 3: Linear Probe Results')
print('='*60)
print(f'  Val->Val   (5-fold GroupCV):  {ex3_lp_vv:.4f}')
print(f'  Val->Train (cross-split):    {ex3_lp_vt:.4f}')
print(f'  Train->Val (cross-split):    {ex3_lp_tv:.4f}')
print(f'  Train->Train (5-fold CV):    {ex3_lp_tt:.4f}')


In [ ]:
# ── Experiment 3: Visualization Artifacts ──
show_run_images(WANDB_ENTITY, WANDB_PROJECT, ex3_eval,
                key_prefix='IntraBag_CosSim', max_images=1, figsize=(8, 7))

show_run_images(WANDB_ENTITY, WANDB_PROJECT, ex3_eval,
                key_prefix='UMAP3_ValSeparability', max_images=1, figsize=(10, 8))

show_run_images(WANDB_ENTITY, WANDB_PROJECT, ex3_eval,
                key_prefix='UMAP1_SubjectFont', max_images=1, figsize=(10, 8))

show_run_images(WANDB_ENTITY, WANDB_PROJECT, ex3_eval,
                key_prefix='UMAP2_DomainShift', max_images=1, figsize=(10, 8))


### Experiment 3 Results & Interpretation
*   **Proxy-A Subject ID Probe:** Crashed to **~8-11%** (Near random chance).
*   **Zero-Shot (Train -> Val):** ~32%
*   **Few-Shot Holdout (Val -> Val):** 83% - 88%
*   **Visual Proof:** The EA Heatmap shows razor-sharp physiological state transitions, proving that the block-diagonal structure is purely cognitive/emotional, not an artifact of shifting cap impedance.

**Interpretation:** EA is a massive success. The Proxy-A crash proves the spatial biometric fingerprint has been mathematically erased. The `Val->Val` probe proves the emotional geometry remains highly intact. 


## 7. The "Tempo" Barrier: Why Zero-Shot is a Myth for 1-Second EEG
If the spatial skull is gone (Proxy-A ~11%), why is the Zero-Shot `Train->Val` accuracy still hovering at ~32%? We assert that **35% - 40% is the biological ceiling for instantaneous 1-second EEG cross-subject emotion recognition.** 

Subject identity in EEG consists of two components:
1. **Spatial Covariance (The Skull):** EA successfully erased this.
2. **Temporal Rhythms (The Tempo):** Individual Alpha Frequency (IAF) and resting baseline hums. Subject A's brain hums at 9Hz; Subject B hums at 11Hz. 

<div align="center">
  <img src="https://media1.tenor.com/m/QDdMxaJsZWUAAAAd/not-quite-my-tempo-jk-simmons.gif" width="400">
  <br>
  <i>The Contrastive Encoder evaluating an unseen subject's Individual Alpha Frequency.</i>
</div>
<br>

Because our CWT preserves the exact 32-timestep sequence, the MobileNet's Temporal Shift Modules (TSM) simply "count the beats." It realizes the tempo has changed, placing the unseen subject in a shifted coordinate space. We *cannot* normalize the temporal axis to fix this, because emotions themselves (e.g., Gamma spikes, Alpha suppression) are shifts in temporal frequency. **You cannot delete a person's temporal rhythm without deleting their emotion.**


## 8. Conclusion & Next Steps (Phase 1B & Beyond)
By Milestone 2, we have achieved a highly robust, spatially-anonymized feature extractor. We have empirically proven two claims that dictate the future of this architecture:
1. 1-second labels are heavily polluted by "Neutral Gaslighting" (proven by the Intra-bag Heatmaps).
2. The latent space contains perfect emotional separability (proven by the 88% `Val->Val` linear probes), but is shifted globally by the subject's biological "Tempo."

To bridge the final temporal domain gap and transition to our Spiking Neural Network (SNN), our roadmap is structured into three upcoming phases:

### Phase 1B: SwiGLU MIL & "Strike-MoCo" Fine-Tuning
To bypass the 24GB VRAM limits of 5D spatiotemporal video sequences, we are adapting the Top-K pruning techniques from Gigapixel Pathology (e.g., CLAM) and merging them with Contrastive Learning to create a custom **Strike-MoCo** architecture:
*   **The Scout Pass (Frozen):** We pass all windows of a 5-minute video through the frozen Phase 1A backbone. The SwiGLU Attention Head scores all windows, identifying the emotional climax and pruning the neutral gaslighting.
*   **The Strike Pass (Unfrozen):** We select only the Top-K climax windows and pass them through an *unfrozen* Stage 4 and Projection Head with `requires_grad=True`. SwiGLU aggregates these into a single Bag-Level Embedding, which is pushed against a MoCo queue of past bags. This perfectly solves the "Blind Scout" paradox while allowing end-to-end temporal fine-tuning without GPU memory explosion.

### Phase 2 & 3: SNN-ANN Distillation
Once Phase 1B finalizes the continuous topological attention masks (, and hopefully makes encoder truly subject-agnostic), we move to the core thesis of the project: **Hardware Efficiency.** 
We will distill the ANN's spatiotemporal knowledge into an **ALIF Spiking U-Net Decoder**. We will aggressively benchmark Synaptic Operations (SynOps) against standard Multiply-Accumulates (MACs) to empirically prove the extreme battery and memory efficiency of our neuromorphic architecture compared to traditional dense networks.

### The Ultimate Pivot: Generative Neuromorphic BCI (Emotion-Subject Synthesizer)
Because Euclidean Alignment (EA) successfully separated the spatial "Skull" from the temporal "Emotion," we have unlocked a massive theoretical pivot for the final product. 
Rather than simply predicting an emotion class, we can treat the SNN Decoder as a **Generative Digital Twin of the Cortex**. 
By taking a "Universal Fear" vector from the latent space, decoding it into a biologically continuous EEG topomap, and multiplying it by the *inverse* EA whitening matrix ($R^{1/2}$) of an unseen subject, we can synthesize highly accurate, biologically plausible EEG trials of a subject experiencing an emotion they never actually watched. This transitions our project from a standard emotion classifier to the first **Neuromorphic Emotion-Subject Synthesizer**.